# Mini-Project 2: Uganda Tax Engine

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [2]:
from datetime import date
from src.tax_engine import PAYE, Money, NoScheduleForDateError, TaxContext

paye = PAYE()
for amount in [335_000, 335_001, 410_000, 485_000, 10_000_000, 10_000_001]:
    result = paye.calculate(TaxContext(on=date(2026, 10, 1), gross_pay=Money(amount)))
    print(f"{amount:>12,}  ->  {result.tax}   ({result.note})")

print(paye.calculate(TaxContext(on=date(2026, 3, 1), gross_pay=Money(10_000_000))).tax,
      "(FY 2025/26 resident at 10,000,000)")
print(paye.calculate(TaxContext(on=date(2026, 10, 1), gross_pay=Money(1_000_000),
                                resident=False)).tax, "(non-resident at 1,000,000)")
try:
    paye.calculate(TaxContext(on=date(2028, 1, 1), gross_pay=Money(500_000)))
except NoScheduleForDateError as error:
    print("NoScheduleForDateError:", error)

     335,000  ->  UGX 0   (Resident, FY 2026/27)
     335,001  ->  UGX 0   (Resident, FY 2026/27)
     410,000  ->  UGX 15,000   (Resident, FY 2026/27)
     485,000  ->  UGX 33,750   (Resident, FY 2026/27)
  10,000,000  ->  UGX 2,888,250   (Resident, FY 2026/27)
  10,000,001  ->  UGX 2,888,250   (Resident, FY 2026/27)
UGX 2,902,000 (FY 2025/26 resident at 10,000,000)
UGX 225,500 (non-resident at 1,000,000)
NoScheduleForDateError: No resident PAYE schedule covers 2028-01-01.


In [3]:
from src.tax_engine import PayrollPipeline

payroll = PayrollPipeline(lst_annual=Money(30_000))   # example annual LST
for month in (date(2026, 10, 1), date(2026, 11, 1)):
    slip = payroll.run(TaxContext(on=month, gross_pay=Money(1_000_000)))
    print(month, "| NSSF", slip.nssf_employee.tax, "| LST", slip.lst.tax,
          "| PAYE", slip.paye.tax, "| net", slip.net_pay,
          "| employer NSSF", slip.nssf_employer.tax)

2026-10-01 | NSSF UGX 50,000 | LST UGX 7,500 | PAYE UGX 186,000 | net UGX 756,500 | employer NSSF UGX 100,000
2026-11-01 | NSSF UGX 50,000 | LST UGX 0 | PAYE UGX 188,250 | net UGX 761,750 | employer NSSF UGX 100,000


In [4]:
from src.tax_engine import CorporateIncomeTax, Invoice, SupplyType, VAT

invoices = (
    Invoice(Money(1_000_000), SupplyType.STANDARD, is_sale=True),
    Invoice(Money(500_000), SupplyType.ZERO_RATED, is_sale=True),
    Invoice(Money(200_000), SupplyType.EXEMPT, is_sale=True),
    Invoice(Money(400_000), SupplyType.STANDARD, is_sale=False),
    Invoice(Money(100_000), SupplyType.STANDARD, is_sale=False, for_exempt_use=True),
    Invoice(Money(50_000), SupplyType.EXEMPT, is_sale=False),
)
vat = VAT()
ctx = TaxContext(on=date(2026, 10, 1), invoices=invoices)
print("Output VAT:", vat.output_vat(invoices), "| Input VAT:", vat.input_vat(invoices),
      "| Net payable:", vat.calculate(ctx).tax)

cit = CorporateIncomeTax()
print("CIT profit:", cit.calculate(TaxContext(on=date(2026, 10, 1),
      revenue=Money(50_000_000), deductions=Money(35_000_000))).tax)
print("CIT loss:  ", cit.calculate(TaxContext(on=date(2026, 10, 1),
      revenue=Money(10_000_000), deductions=Money(12_000_000))).tax)

Output VAT: UGX 180,000 | Input VAT: UGX 72,000 | Net payable: UGX 108,000
CIT profit: UGX 4,500,000
CIT loss:   UGX 0


In [5]:
from src.tax_engine import PaymentType, RentalIncomeTax, WithholdingTax

on = date(2026, 10, 1)
fees = WithholdingTax(PaymentType.PROFESSIONAL_FEES)
goods = WithholdingTax(PaymentType.GOODS_AND_SERVICES)
print("Fees 2,000,000, resident:     ", fees.calculate(TaxContext(on=on, revenue=Money(2_000_000))).tax)
print("Fees 2,000,000, non-resident: ", fees.calculate(TaxContext(on=on, revenue=Money(2_000_000), resident=False)).tax)
print("Goods 800,000 (below 1m):     ", goods.calculate(TaxContext(on=on, revenue=Money(800_000))).tax)
print("Goods 1,500,000:              ", goods.calculate(TaxContext(on=on, revenue=Money(1_500_000))).tax)

print("Rental, individual 6,000,000: ", RentalIncomeTax().calculate(TaxContext(on=on, revenue=Money(6_000_000))).tax)
print("Rental, company 30m, exp 20m: ", RentalIncomeTax(individual=False).calculate(
      TaxContext(on=on, revenue=Money(30_000_000), deductions=Money(20_000_000))).tax)

Fees 2,000,000, resident:      UGX 120,000
Fees 2,000,000, non-resident:  UGX 300,000
Goods 800,000 (below 1m):      UGX 0
Goods 1,500,000:               UGX 90,000
Rental, individual 6,000,000:  UGX 381,600
Rental, company 30m, exp 20m:  UGX 4,500,000
